In [11]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.feature_selection import RFE


def data_report(file_name):
    fpath='/home/adityapandey/Downloads/project-sem5/Data Sets/'
    from ydata_profiling import ProfileReport
    df = pd.read_csv(fpath+file_name)
    profile = ProfileReport(df, title="final report1", explorative=True)
    profile.to_file("predict_data.html")

def load_data():
    fpath='/home/adityapandey/Downloads/project-sem5/Data Sets/'
    reputation = pd.read_csv(fpath+"QS World University Rankings 2025 (Top global universities).csv",encoding='latin1')
    predict= pd.read_csv(fpath+"admission_data.csv")
    return reputation,predict

reputation, predict = load_data()
reputation.columns = reputation.columns.str.strip().str.lower().str.replace(" ", "_")
predict.columns = predict.columns.str.strip().str.lower().str.replace(" ", "_")
reputation=reputation.drop(columns=['overall_score','rank_2024','region','size','focus','res.','status','academic_reputation_score', 'academic_reputation_rank', 'employer_reputation_score', 'employer_reputation_rank', 'faculty_student_score', 'faculty_student_rank', 'citations_per_faculty_score', 'citations_per_faculty_rank', 'international_faculty_score', 'international_faculty_rank', 'international_students_score', 'international_students_rank', 'international_research_network_score', 'international_research_network_rank', 'employment_outcomes_score', 'employment_outcomes_rank', 'sustainability_score', 'sustainability_rank'])
reputation["university_rating"]=1
reputation['rank_2025'] = pd.to_numeric(reputation['rank_2025'], errors='coerce')
reputation = reputation[reputation['rank_2025'] <= 800]
reputation = reputation.drop(reputation.loc[reputation['location'] == 'United Arab Emirates'].index)
reputation = reputation.drop(reputation.loc[reputation['location'] == 'South Korea'].index)
reputation = reputation.drop(reputation.loc[reputation['location'] == 'India'].index)
reputation.loc[reputation['rank_2025'] <= 100, 'university_rating'] = 5
reputation.loc[(reputation['rank_2025'] > 100) & (reputation['rank_2025'] <= 200), 'university_rating'] = 4
reputation.loc[(reputation['rank_2025'] > 200) & (reputation['rank_2025'] <= 300), 'university_rating'] = 3
reputation.loc[(reputation['rank_2025'] > 300) & (reputation['rank_2025'] <= 400), 'university_rating'] = 2
reputation.loc[reputation['rank_2025'] > 400, 'university_rating'] = 1

'''df=reputation.merge(predict, how='inner', on='university_rating')
fpath='/home/adityapandey/Downloads/project-sem5/Data Sets/'
df.to_csv(fpath+"merged_university_data.csv", index=False)
data_report("merged_university_data.csv")'''

'df=reputation.merge(predict, how=\'inner\', on=\'university_rating\')\nfpath=\'/home/adityapandey/Downloads/project-sem5/Data Sets/\'\ndf.to_csv(fpath+"merged_university_data.csv", index=False)\ndata_report("merged_university_data.csv")'

In [12]:
predict_data = df.copy()
predict_data=predict_data.drop(columns=['rank_2025','institution_name','location'])
x_train,x_test,y_train,y_test=train_test_split(predict_data.drop(columns=['chance_of_admit']),predict_data['chance_of_admit'],test_size=0.2,random_state=42)
scaler=MinMaxScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)


# Create polynomial features (degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(x_train_scaled)
X_test_poly = poly.transform(x_test_scaled)

score=[]
for i in range(1,X_train_poly.shape[1]+1):
    ref=RFE(LinearRegression(),n_features_to_select=i)
    ref.fit(X_train_poly,y_train)
    r2=r2_score(y_test,ref.predict(X_test_poly))
    score.append([r2,i])


In [13]:
ref=RFE(LinearRegression(),n_features_to_select=15)
ref.fit(X_train_poly,y_train)

y_train_predict=ref.predict(X_train_poly)
y_test_predict=ref.predict(X_test_poly)
print("Train R2 Score:",r2_score(y_train,y_train_predict))
print("Test R2 Score:",r2_score(y_test,y_test_predict))
coefficients=ref.estimator_.coef_
intrecept=ref.estimator_.intercept_
print("Intercept:",intrecept)
print("Coefficients:",coefficients)




Train R2 Score: 0.8433532992637206
Test R2 Score: 0.8451547196210989
Intercept: 0.31742242977572266
Coefficients: [-0.11531787  0.13136665  0.08158477  0.21031592  0.44270259  0.05705684
  0.34506049 -0.15800486  0.23740221 -0.31484863  0.19481417 -0.238126
 -0.22366765 -0.24960554  0.28829949]
